In [1]:
# Import standard libraries
import sys
import re

sys.path.append("..")

import phcpy as hc  # Homotopy continuation package
from phcpy.dimension import get_core_count # For getting number of CPU cores
from phcpy.solver import solve # For solving polynomial systems
from phcpy.solutions import filter_real # For filtering real solutions

import sympy as sp # For symbolic mathematics
from sympy.parsing.sympy_parser import parse_expr

import matplotlib.pyplot as plt # For plotting
import plotly.graph_objects as go

# Import our modules
from src.model import *
from src.nnexpansion import *
from src.utils import *

from src.edop import *

PHCv2.4.90 released 2024-03-20 works!


### Part 1
We at first generate a synthetic dataset with a noisy sinusoidal decision boundary. The dataset is visualized below, with blue and orange points representing the two classes.

In [2]:
# Set random seed for reproducibility
torch.manual_seed(42)

# Generate training data
print("Generating training data with sin(x) decision boundary...")
X_train, y_train = generate_sin_boundary_data(n_samples=2000, noise_level=0.1)

print(f"Training data shape: {X_train.shape}")
print(f"Labels shape: {y_train.shape}")
print(f"Class distribution: {torch.bincount(y_train.long())}")

Generating training data with sin(x) decision boundary...


Training data shape: torch.Size([2000, 2])
Labels shape: torch.Size([2000])
Class distribution: tensor([1019,  981])


We then train a shallow neural network to classify the data. The network architecture consists of two hidden layers with learnable parameters in the activation functions. The first hidden layer has 8 units, and the second hidden layer has 6 units. The activation functions are degree 3 polynomials. After training, we visualize the learned decision boundary of the neural network, which approximates the true sinusoidal boundary.

In [3]:
# Create polynomial network
model = PolynomialNetwork(
    input_dim=2,
    output_dim=2,  # Binary classification with 2 outputs
    hidden_dims=[8, 6],
    polynomial_degree=3,
)

print(f"\nModel architecture:")
print(f"Input dim: {model.input_dim}")
print(f"Hidden dims: {model.hidden_dims}")
print(f"Output dim: {model.output_dim}")
print(f"Polynomial degree: {model.polynomial_degree}")


Model architecture:
Input dim: 2
Hidden dims: [8, 6]
Output dim: 2
Polynomial degree: 3


In [4]:
# Train the model
print("\nTraining the model...")
losses, accuracies = train_model(model, X_train, y_train, epochs=3000, lr=1e-3)


Training the model...


Training Progress:   0%|          | 0/3000 [00:00<?, ?it/s]

In [5]:
# Visualize decision boundary
visualize_decision_boundary(
    model, X_train, y_train, x_range=(-10, 10), y_range=(-10, 10)
)

### Part 2
Next, we perform a polynomial expansion of the trained neural network. We can then compute the implicit function that defines the decision boundary of the neural network. This implicit function is a polynomial equation in two variables (x0, x1).

In [6]:
# Monomial Expansion
polys, monoms, C = polynomial_nn_expansion(model)
print("Number of monomials: ", len(monoms))
print("Shape of Coefficient matrix:", C.shape)

print("Monomials:", monoms)

Input symbols: (x0, x1)
Layer 0 weight: (8, 2)
Layer 0 bias: (8, 1)
Layer 0 activation: (4,)
Layer 1 weight: (6, 8)
Layer 1 bias: (6, 1)
Layer 1 activation: (4,)
Layer 2 weight: (2, 6)
Layer 2 bias: (2, 1)
Layer 2 activation: None
Polynomial expansion took 0.7840 seconds.
Number of monomials:  55
Shape of Coefficient matrix: (2, 55)
Monomials: [(0, 0), (0, 1), (1, 0), (0, 2), (1, 1), (2, 0), (0, 3), (1, 2), (2, 1), (3, 0), (0, 4), (1, 3), (2, 2), (3, 1), (4, 0), (0, 5), (1, 4), (2, 3), (3, 2), (4, 1), (5, 0), (0, 6), (1, 5), (2, 4), (3, 3), (4, 2), (5, 1), (6, 0), (0, 7), (1, 6), (2, 5), (3, 4), (4, 3), (5, 2), (6, 1), (7, 0), (0, 8), (1, 7), (2, 6), (3, 5), (4, 4), (5, 3), (6, 2), (7, 1), (8, 0), (0, 9), (1, 8), (2, 7), (3, 6), (4, 5), (5, 4), (6, 3), (7, 2), (8, 1), (9, 0)]


In [7]:
# Implicit polynomial equation for decision boundary
poly_decision = polys[1] - polys[0]
print("Implicit polynomial equation for decision boundary:")
poly_decision

Implicit polynomial equation for decision boundary:


-4.88198798349657e-6*x0**9 + 8.75888708880666e-5*x0**8*x1 - 0.000284459204586999*x0**8 + 0.000322741456345137*x0**7*x1**2 - 0.000135255272251752*x0**7*x1 - 0.000927094158530457*x0**7 + 0.0001071211273888*x0**6*x1**3 - 0.00352488488224806*x0**6*x1**2 - 0.00693980045801572*x0**6*x1 + 0.0141487326687443*x0**6 - 0.00081867898969386*x0**5*x1**4 - 0.00333451158420848*x0**5*x1**3 - 0.00367075154867362*x0**5*x1**2 - 0.002130046407121*x0**5*x1 + 0.0159371041995068*x0**5 + 0.00325253954928057*x0**4*x1**5 + 0.0170193665376538*x0**4*x1**4 + 0.0228772323463813*x0**4*x1**3 + 0.127332798103214*x0**4*x1**2 + 0.310568692241639*x0**4*x1 - 0.163164868122971*x0**4 - 0.00385160773942667*x0**3*x1**6 - 0.0076733151656452*x0**3*x1**5 + 0.122782406995632*x0**3*x1**4 + 0.363591626295294*x0**3*x1**3 + 0.0315357999268162*x0**3*x1**2 + 0.404223966632909*x0**3*x1 + 1.50888654369427*x0**3 + 0.00876103355905843*x0**2*x1**7 + 0.0782986495304738*x0**2*x1**6 + 0.114598098049775*x0**2*x1**5 - 0.46644116792397*x0**2*x1**4

In [8]:
# Solve for critical points
best_critical_point, gamma, real_sols = get_critical_points(np.array([-10.0, -5.0]), poly_decision, model)

Using 31 CPU cores for homotopy continuation.
Solved in 0.08 seconds.
Found 7 real solutions out of 81 complex solutions.
Removing duplicates, 7 unique real solutions remain.
Closest critical point: [-7.78130306  0.2879432 ]
Distance to closest critical point (gamma): 5.734540908729725


In [9]:
# Solve for critical points
# best_critical_point, gamma, real_sols = get_critical_points(np.array([1.0, -5.0]), poly_decision, model)